# Plot Bias Corrected Annual Mean Calculated PM<sub>2.5</sub>

Using CESM2 SSP2-4.5 ensemble 1 as an example

In [ ]:
import os
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FuncFormatter
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from utils.utils import autosize_figure, land_filter, get_scenario_config
import config
from utils.utils import require_dir
import pathlib

In [ ]:
def process_pm25_for_plotting(model, scenario, years, ens_num):
    # === Path config ===
    PM25_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "pm25" / "annual_pm25_bc")

    dates = f"{years.start}-{years.stop}"

    file = f"Annual_PM25_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    path = os.path.join(PM25_DIR, file)
    da = xr.open_dataarray(path)

    # Calculate end of scenario mean (10 years)
    end_idx = da.sizes["year"]
    # Select the last 10 years
    da_last10 = da.isel(year=slice(end_idx - 10, end_idx))

    # Get the first and last time values
    start_time = da_last10.year[0].item()
    end_time = da_last10.year[-1].item()

    # Calculate temporal mean
    da_mean = da_last10.mean("year")
    return da_mean, start_time, end_time

In [ ]:
def plot_annual_pm25_bc(da, model, scenario, ens_num, start_year, end_year, SAVE_DIR, land=False):
    # Create figure and layout
    fig = plt.figure(figsize=autosize_figure(1, 1))
    gs = GridSpec(2, 1, height_ratios=[15, 1])

    # Map projection and display
    projection = ccrs.Robinson()
    crs = ccrs.PlateCarree()

    country_borders = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_0_boundary_lines_land',
        scale='50m',
        facecolor='none')

    if land is True:
        da = land_filter(da)

    # Use log scale for normalization
    vmax = da.max().item()
    cmap = plt.get_cmap("magma")

    # Define a linear threshold region around zero
    linthresh = 1

    norm = SymLogNorm(linthresh=linthresh, vmin=0, vmax=vmax, base=10)

    # First panel
    ax = fig.add_subplot(gs[0, 0], projection=projection, frameon=True)
    cb = da.plot(
        transform=crs,
        add_colorbar=False,
        cmap=cmap,
        norm=norm,
        subplot_kws={'projection': projection}
    )
    ax.coastlines(resolution="50m", linewidth=0.75)
    ax.add_feature(country_borders, edgecolor='k', linewidth=0.75)
    plt.title(f"Bias Corrected Annual PM2.5 \n {model} {scenario} {start_year}-{end_year}", fontsize=16)

    # Colorbar
    cax = fig.add_subplot(gs[1, 0])
    col_bar = plt.colorbar(cb, cax=cax, orientation='horizontal')
    col_bar.set_label("PM2.5 (μg/m3)", fontsize=13)

    # Ticks
    formatter = FuncFormatter(lambda v, _: f"{v:g}")
    col_bar.ax.xaxis.set_major_formatter(formatter)

    plt.tight_layout()

    if land is True:
        out_file = f"Annual_PM25_BC_land_{model}_{scenario}_{ens_num:02d}_{start_year}-{end_year}.png"
    else:
        out_file = f"Annual_PM25_BC_{model}_{scenario}_{ens_num:02d}_{start_year}-{end_year}.png"
    out_path = os.path.join(SAVE_DIR, out_file)
    plt.savefig(out_path)
    return

In [ ]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
SAVE_DIR = require_dir(pathlib.Path(config.PLOTTING_ROOT) / "pm25" / "Annual_PM25_BC")

model = "CESM2"
scenario = "SSP245"
ensemble_number = 1

configs = get_scenario_config(model, scenario)
years = configs["years"]

da, first_year, last_year = process_pm25_for_plotting(model, scenario, years, ensemble_number)

# Plotting without ocean
plot_annual_pm25_bc(da, model, scenario, ensemble_number, first_year, last_year, SAVE_DIR, land=True)
# Plotting with ocean
plot_annual_pm25_bc(da, model, scenario, ensemble_number, first_year, last_year, SAVE_DIR, land=False)